# **Cell Convert & Split**

In [ ]:
import os
import json
import glob
import random
from shapely import wkt
from tqdm import tqdm

random.seed(42)

# --- Konfigurasi ---
TRAIN_BASE  = '/kaggle/input/datasets/trsnwn/xbd-train/train'
TEST_BASE   = '/kaggle/input/datasets/trsnwn/xbd-test1/test'
OUTPUT_BASE = '/kaggle/working/xbd_yolo'
IMG_SIZE    = 1024

DAMAGE_MAP = {'minor-damage': 0, 'major-damage': 1, 'destroyed': 2}

for split in ['train', 'val', 'test']:
    os.makedirs(f'{OUTPUT_BASE}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUTPUT_BASE}/{split}/labels', exist_ok=True)


def polygon_wkt_to_bbox_yolo(wkt_str):
    poly = wkt.loads(wkt_str)
    minx, miny, maxx, maxy = poly.bounds
    cx = ((minx + maxx) / 2) / IMG_SIZE
    cy = ((miny + maxy) / 2) / IMG_SIZE
    bw = (maxx - minx) / IMG_SIZE
    bh = (maxy - miny) / IMG_SIZE
    return cx, cy, bw, bh


def convert_split(base_path, split_name, val_ratio=0.2):
    label_dir = os.path.join(base_path, 'labels')
    image_dir = os.path.join(base_path, 'images')

    json_files = sorted(glob.glob(os.path.join(label_dir, '*_post_disaster.json')))
    print(f"\n[{split_name}] Ditemukan {len(json_files)} file JSON")

    if split_name == 'train':
        random.shuffle(json_files)
        cut = int(len(json_files) * (1 - val_ratio))
        subsets = [('train', json_files[:cut]), ('val', json_files[cut:])]
    else:
        subsets = [('test', json_files)]

    # hitung jumlah objek tiap kelas per subset
    counts = {out_split: {k: 0 for k in DAMAGE_MAP} for out_split, _ in subsets}

    for out_split, files in subsets:
        for json_path in tqdm(files, desc=f'  -> {out_split}'):
            with open(json_path) as f:
                data = json.load(f)

            basename     = os.path.basename(json_path).replace('_post_disaster.json', '')
            img_filename = basename + '_post_disaster.png'
            img_src      = os.path.join(image_dir, img_filename)

            yolo_lines = []
            for feat in data['features']['xy']:
                subtype = feat['properties']['subtype'].strip().lower()
                if subtype not in DAMAGE_MAP:
                    continue
                cx, cy, bw, bh = polygon_wkt_to_bbox_yolo(feat['wkt'])
                yolo_lines.append(f"{DAMAGE_MAP[subtype]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                counts[out_split][subtype] += 1

            txt_path = f"{OUTPUT_BASE}/{out_split}/labels/{basename}_post_disaster.txt"
            with open(txt_path, 'w') as f:
                f.write('\n'.join(yolo_lines))

            img_dst = f"{OUTPUT_BASE}/{out_split}/images/{img_filename}"
            if not os.path.exists(img_dst):
                os.symlink(img_src, img_dst)

    # cetak distribusi kelas tiap subset
    for out_split in counts:
        print(f"\n  Distribusi kelas ({out_split}):")
        for cls, n in counts[out_split].items():
            print(f"    {cls:<15}: {n:,}")


convert_split(TRAIN_BASE, 'train')
convert_split(TEST_BASE, 'test')
print("\nKonversi selesai.")

In [ ]:
!pip install ultralytics

# **Cell Train Scratch**

In [ ]:
from ultralytics import YOLO

DATA_YAML   = "/kaggle/working/xbd_yolo/data.yaml"
PROJECT_DIR = "/kaggle/working/runs"
RUN_NAME    = "yolov8n_c2f_scratch"

with open(DATA_YAML, "w") as f:
    f.write("""path: /kaggle/working/xbd_yolo
train: train/images
val: val/images
test: test/images

nc: 3
names: ['minor-damage', 'major-damage', 'destroyed']
""")

# bukan dari .pt pre-trained -> training from scratch
model = YOLO("yolov8n.yaml")

results = model.train(
    data=DATA_YAML,
    project=PROJECT_DIR,
    name=RUN_NAME,

    epochs=100,
    patience=0,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.0001,

    fliplr=0.5,
    flipud=0.5,
    scale=0.5,
    mosaic=1.0,
    mixup=0.1,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    translate=0.0,

    save=True,
    save_period=-1,
    plots=True,
    verbose=True,
    exist_ok=True,
)

print(f"\nTraining selesai. Hasil tersimpan di: {PROJECT_DIR}/{RUN_NAME}")

# **Cell Test Set**

In [ ]:
from ultralytics import YOLO

RUN_DIR = '/kaggle/working/runs/yolov8n_scratch'
CLASS_NAMES = ['minor-damage', 'major-damage', 'destroyed']
IOU_THRESHOLD = 0.5   # sama dengan threshold mAP50

model = YOLO(f'{RUN_DIR}/weights/best.pt')
metrics = model.val(
    data='/kaggle/working/xbd_yolo/data.yaml',
    split='test',
    imgsz=640,
    iou=IOU_THRESHOLD,
)

cm = metrics.confusion_matrix.matrix
iou_per_class = []
for i in range(len(CLASS_NAMES)):
    tp = cm[i, i]
    fp = cm[i, :].sum() - tp
    fn = cm[:, i].sum() - tp
    denom = tp + fp + fn
    iou = (tp / denom) if denom > 0 else 0.0
    iou_per_class.append(iou)

# Cetak hasil per kelas, langsung dari metrics.box (bawaan Ultralytics)
print(f"\nHasil Evaluasi Test Set: default_augmentation")
for i, kelas in enumerate(CLASS_NAMES):
    print(f"{kelas:<15} | Precision={metrics.box.p[i]:.4f}  Recall={metrics.box.r[i]:.4f}  "
          f"mAP50={metrics.box.ap50[i]:.4f}  mAP50-95={metrics.box.maps[i]:.4f}  "
          f"F1={metrics.box.f1[i]:.4f}  IoU={iou_per_class[i]:.4f}")

print(f"{'Rata-rata':<15} | Precision={metrics.box.p.mean():.4f}  Recall={metrics.box.r.mean():.4f}  "
      f"mAP50={metrics.box.ap50.mean():.4f}  mAP50-95={metrics.box.maps[:len(CLASS_NAMES)].mean():.4f}  "
      f"F1={metrics.box.f1.mean():.4f}  IoU={sum(iou_per_class) / len(iou_per_class):.4f}")

# **Cell Print Curve & Confusion Matrix**

In [ ]:
from IPython.display import Image, display
import os

RUN_DIR = '/kaggle/working/runs/yolov8n_scratch'

for fname in ['results.png', 'PR_curve.png', 'F1_curve.png',
              'confusion_matrix_normalized.png']:
    fpath = os.path.join(RUN_DIR, fname)
    if os.path.exists(fpath):
        print(fname)
        display(Image(fpath))